# SelfConditionedDenoisingAtoms — Examples

Three minimal examples:
1. **Load a pretrained model from HuggingFace**
2. **Instantiate a new untrained model**
3. **Forward pass with a QM9 sample**

## Example 1: Load a Pretrained Model from HuggingFace

Available pretrained models: `"ct-scd-pcq"`, `"ct-scd-amp"`

The checkpoint is downloaded from `Ty-Perez/<model_name>` on HuggingFace Hub and the bare `model` (`CET` or similar) is returned — ready for inference or fine-tuning.

In [ ]:
from huggingface_hub import hf_hub_download
from models.model_helper2 import load_model

# ── choose a pretrained model ──────────────────────────────────────────────────
MODEL_NAME  = "ct-scd-pcq"          # or "ct-scd-amp"
REPO_PREFIX = "Ty-Perez/"
SAVE_DIR    = "./experiments"        # directory must already exist

# 1. Download the checkpoint from HuggingFace (cached after first download)
ckpt_path = hf_hub_download(
    repo_id=f"{REPO_PREFIX}{MODEL_NAME}",
    filename="last.ckpt",
    cache_dir=SAVE_DIR,
)
print(f"Checkpoint at: {ckpt_path}")

# 2. Load the model weights
model, ema_model, ckpt = load_model(ckpt_path)
model.eval()

#print number of parameters in the model (in the millions)
num_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Number of parameters: {num_params:.2f} M")
print(model)

'''
NOTE: for the pretrained models on huggingface the ema_model is NOT trained. 
use the model for inference, not the ema_model. the training script provided 
also allows for training an ema_model needed, but we did not find it to be 
necessary for good performance.
'''


## Example 2: Instantiate a New (Untrained) Model

`create_model` reads a model-config YAML and constructs the network from scratch with random weights. The YAML specifies the architecture (e.g. `CET`) and all hyper-parameters.

In [ ]:
from models.model_helper2 import create_model

# Minimal args dict — only model_config is required.
# Any key present in the YAML can be overridden here.
args = {
    "model_config": "configs/model_configs/scd.yaml",  # architecture + hyper-params
    "prior_model": None,   # no prior (e.g. Atomref) by default
}

model = create_model(args)
model.eval()
print(model)

## Example 3a: Forward Pass without torchnet graph kernel

In [18]:
import torch
from data.datasets import QM9
from torch_geometric.data import Batch
from data.datasets.transforms import AddStandardKeys
from torch_geometric.data import DataLoader

#the AddStandardKeys Transform create the necessary keys for graph creation and standardization across different kinds of samples
dataset = QM9(root="tmp/qm9", dataset_arg="homo", transform=AddStandardKeys())
sample = dataset[0]
loader = DataLoader(dataset, batch_size=16, shuffle=True)
data_batch = next(iter(loader))

model.eval()
with torch.no_grad():
    out = model(
        z=data_batch.z,
        pos=data_batch.pos,
        batch=data_batch.batch,
        graph_batch=data_batch,
    )

print("Forward output keys:", sorted(out.keys()))
print("y shape:", tuple(out["y"].shape))


Forward output keys: ['mol_emb', 'noise_pred', 'y']
y shape: (16, 1)


/tmp/ipykernel_820597/3738888321.py:10: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  loader = DataLoader(dataset, batch_size=16, shuffle=True)


## Example 3: Forward Pass on a QM9 Sample with Torchnet graph kernel

Load a single molecule from the QM9 dataset and run it through the HuggingFace-loaded model.

The model's `forward(z, pos, batch)` expects:
- `z` — atomic numbers `(N,)`
- `pos` — Cartesian coordinates `(N, 3)`
- `batch` — molecule index per atom `(N,)` (optional for a single molecule)

It returns a dict with keys `"y"` (scalar prediction), `"noise_pred"`, `"mol_emb"`, etc.

In [ ]:
import torch
from data.datasets import QM9
from torch_geometric.data import Batch

# ── 1. Load QM9 and grab one molecule ─────────────────────────────────────────
# dataset_arg selects the target property (see QM9.target_dict_custom for all options)
dataset = QM9(root="tmp/qm9", dataset_arg="homo")
sample = dataset[0]   # a single torch_geometric Data object

print("Atomic numbers (z):", sample.z)
print("Positions shape:   ", sample.pos.shape)   # (N, 3)
print("Num atoms:         ", sample.z.shape[0])
print("Target (homo):     ", sample.y)

# ── 2. Run the forward pass ───────────────────────────────────────────────────
# Uses the pretrained model loaded in Example 1.
# batch can be omitted for a single molecule (defaults to all zeros inside the model).
model.eval()
with torch.no_grad():
    out = model(z=sample.z, pos=sample.pos)

print("\nOutput keys:", list(out.keys()))
print("Scalar prediction (y):", out["y"])   # eV (same units as homo target)

# ── 3. Batch multiple molecules ───────────────────────────────────────────────
batch = Batch.from_data_list([dataset[i] for i in range(4)])

with torch.no_grad():
    out_batch = model(z=batch.z, pos=batch.pos, batch=batch.batch)

print("\nBatched scalar predictions (y):", out_batch["y"])  # shape (4,)

#print keys in out_batch
print("Batched output keys:", list(out_batch.keys()))